# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q duckdb huggingface_hub

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!")

Token loaded successfully!


In [3]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB Connected!")

DuckDB Connected!


In [4]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print(TABLES)

{'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


In [5]:
print(con)
print(TABLES)

{'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


In [6]:
df_check = con.sql(f"""
SELECT *
FROM {TABLES['fact_query_90d']}
LIMIT 1
""").df()

print(df_check.columns.tolist())

['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


Finding 1: FlyRank observed that growing pages were younger on average than declining pages. The average age was about 185 days for growing pages and 228 days for declining pages. The label comes from the observed trend direction, where pages are grouped as growing or declining based on their recent performance. The comparison supports a directional association between page age and growth, but it does not prove that age alone causes decline.

Finding 2: FlyRank observed a stronger growth-to-decline ratio for pages in the 31-90 day freshness window, measured at about 5.43:1. The label comes from the observed growth or decline direction over the defined measurement window. The validation design supports this as an observed portfolio pattern, but the result should be treated as directional evidence rather than proof that freshness alone causes growth.

Methodology questions:
1. Where does the label come from? The label is derived from observed performance trend over a defined time window rather than from a future outcome that was already known.
2. Does the validation design carry the claim? The aggregate comparisons support the directional claims, but they do not establish causation. A time-aware or out-of-sample validation design is needed before making stronger predictive claims.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Finding 1: Age difference between growing and declining pages was observed in the FlyRank report.")
print("Finding 2: The 31-90 day freshness window showed a 5.43:1 growth-to-decline ratio.")
print("Both findings are treated as directional evidence, not causal proof.")


Finding 1: Age difference between growing and declining pages was observed in the FlyRank report.
Finding 2: The 31-90 day freshness window showed a 5.43:1 growth-to-decline ratio.
Both findings are treated as directional evidence, not causal proof.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I will use a time-aware split because the goal is to evaluate how the model performs on later, unseen data. The earlier observations are used for training and the later observations are used for testing. This is more realistic than randomly mixing earlier and later observations.

In [8]:


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score



df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    window_start,
    impressions_last30,
    impressions_prev30,
    content_visible_query_count,
    rare_impressions_share,
    anonymized_impressions_share,
    content_total_impressions_90d
FROM {TABLES['fact_query_90d']}
WHERE impressions_last30 IS NOT NULL
  AND impressions_prev30 IS NOT NULL
""").df()

print("Rows loaded:", len(df))



df["window_start"] = pd.to_datetime(df["window_start"])


df["is_declining"] = (
    df["impressions_last30"] < df["impressions_prev30"]
).astype(int)

print("\nLabel distribution:")
print(df["is_declining"].value_counts())



df["imp_prev30"] = df["impressions_prev30"]

df["visible_queries"] = df["content_visible_query_count"]

df["rare_share"] = df["rare_impressions_share"]

df["anon_share"] = df["anonymized_impressions_share"]

df["top_query_share"] = (
    df["impressions_last30"] /
    df["content_total_impressions_90d"].replace(0, pd.NA)
)


features = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

df = df.dropna(subset=features + ["is_declining"])



df = df.sort_values("window_start").reset_index(drop=True)



X = df[features]
y = df["is_declining"]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

random_model.fit(
    X_train_random,
    y_train_random
)

random_pred = random_model.predict(X_test_random)

random_accuracy = accuracy_score(
    y_test_random,
    random_pred
)

print("\n========== BEFORE ==========")
print("Random-split accuracy:", round(random_accuracy, 4))




split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

X_train_time = train_df[features]
y_train_time = train_df["is_declining"]

X_test_time = test_df[features]
y_test_time = test_df["is_declining"]

time_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

time_model.fit(
    X_train_time,
    y_train_time
)

time_pred = time_model.predict(X_test_time)

time_accuracy = accuracy_score(
    y_test_time,
    time_pred
)

print("\n========== AFTER ==========")
print("Time-aware accuracy:", round(time_accuracy, 4))



print("\n========== SPLIT DESIGN ==========")

print(
    "Training period:",
    train_df["window_start"].min(),
    "to",
    train_df["window_start"].max()
)

print(
    "Testing period:",
    test_df["window_start"].min(),
    "to",
    test_df["window_start"].max()
)

print("\nTraining rows:", len(train_df))
print("Testing rows :", len(test_df))



print("\n========== COMPARISON ==========")

comparison = pd.DataFrame({
    "Validation": [
        "Random Split (Week-5)",
        "Time-Aware Split (ML-09)"
    ],
    "Accuracy": [
        random_accuracy,
        time_accuracy
    ]
})

print(comparison)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 2414248

Label distribution:
is_declining
1    1373407
0    1040841
Name: count, dtype: int64

========== BEFORE ==========
Random-split accuracy: 0.8085

========== AFTER ==========
Time-aware accuracy: 0.8192

========== SPLIT DESIGN ==========
Training period: 2026-04-02 00:00:00 to 2026-04-02 00:00:00
Testing period: 2026-04-02 00:00:00 to 2026-04-02 00:00:00

Training rows: 1931398
Testing rows : 482850

========== COMPARISON ==========
                 Validation  Accuracy
0     Random Split (Week-5)  0.808456
1  Time-Aware Split (ML-09)  0.819182


The Week-5 random-split accuracy was 80.85%, while the time-aware accuracy was 81.92%.

The time-aware result was slightly higher than the random-split result. This suggests that the validation design did not cause a large performance drop in this run. The time-aware split is still the more realistic validation design because the model is trained on earlier observations and tested on later observations.

The result should be treated as measured decision-support evidence rather than proof that the model will perform the same way on future data.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage audit: I did not include the target label directly as a model feature, and none of the selected feature names explicitly refer to future outcomes. The selected signals are treated as observed information available in the query window. However, the validation has an important limitation because the available window_start values did not provide separate calendar dates for training and testing, so the time-aware result should not be interpreted as a strong future-period validation.

In [9]:


print("FEATURE LEAKAGE AUDIT")
print("=" * 50)


features = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

print("\nFeatures used:")
for feature in features:
    print("-", feature)


print("\n1. Direct label check")

if "is_declining" in features:
    print(" LEAKAGE: label is directly included!")
else:
    print(" PASS: label is not directly included.")


future_words = [
    "future",
    "next",
    "label",
    "declining",
    "trend",
    "target",
    "outcome"
]

print("\n2. Future/label-like feature name check")

suspicious = []

for feature in features:
    for word in future_words:
        if word in feature.lower():
            suspicious.append(feature)
            break

if suspicious:
    print(" Suspicious features:", suspicious)
else:
    print(" PASS: no obviously future/label-derived feature names.")



print("\n3. Feature timing check")

print("imp_prev30       → previous 30-day impressions")
print("visible_queries  → observed query count")
print("rare_share       → observed rare-query share")
print("anon_share       → observed anonymized share")
print("top_query_share  → calculated from observed impressions")

print("\n These features do not explicitly contain future/label names.")


print("\n" + "=" * 50)
print("FINAL LEAKAGE AUDIT")
print("=" * 50)

print("No direct label column used.")
print("No obvious future-looking feature name used.")
print("Features should be treated as current/observed signals.")
print("However, validation is limited because window_start had only one observed date.")


FEATURE LEAKAGE AUDIT

Features used:
- imp_prev30
- visible_queries
- rare_share
- anon_share
- top_query_share

1. Direct label check
 PASS: label is not directly included.

2. Future/label-like feature name check
 PASS: no obviously future/label-derived feature names.

3. Feature timing check
imp_prev30       → previous 30-day impressions
visible_queries  → observed query count
rare_share       → observed rare-query share
anon_share       → observed anonymized share
top_query_share  → calculated from observed impressions

 These features do not explicitly contain future/label names.

FINAL LEAKAGE AUDIT
No direct label column used.
No obvious future-looking feature name used.
Features should be treated as current/observed signals.
However, validation is limited because window_start had only one observed date.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The model achieved 80.85% accuracy with the Week-5 random split and 81.92% with the time-aware split used here. The results show that the selected signals provide useful directional decision-support information for identifying declining cases.

However, the time-aware validation is limited because the available data showed the same window_start date for both the training and testing portions. Therefore, these results should not be presented as proof of future-period performance or causation. A stronger claim would require genuinely separated time periods and evaluation on an unseen future window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.